In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve   # <-- FIXED IMPORT

from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("/Users/juliangendreau/Desktop/labels.csv")

features = [
    "age", "sex_x", "hispanic", "marriage",
    "white_race", "black_race", "other_race"
]
target = "MGMT_methylation"

X = df[features]
y = df[target]

numeric_features = ["age"]
categorical_features = [
    "sex_x", "hispanic", "marriage",
    "white_race", "black_race", "other_race"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        max_iter=1000,
        random_state=42
    ))
])

# -----------------------------
# Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC: {test_auc:.3f}")

# ============================================================
# 1. ROC Curve (Test Set)
# ============================================================
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {test_auc:.3f}", linewidth=2)
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Test Set)")
plt.legend()
plt.tight_layout()
plt.savefig("figure_roc_test.png", dpi=300)
plt.close()

# ============================================================
# 2. Cross‑validated ROC curves
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_cv_pred = cross_val_predict(
    model, X, y, cv=cv, method="predict_proba"
)[:, 1]

cv_auc = roc_auc_score(y, y_cv_pred)
fpr_cv, tpr_cv, _ = roc_curve(y, y_cv_pred)

plt.figure(figsize=(6, 6))
plt.plot(fpr_cv, tpr_cv, label=f"CV AUC = {cv_auc:.3f}", linewidth=2)
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross‑validated ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig("figure_roc_cv.png", dpi=300)
plt.close()

# ============================================================
# 3. Calibration curve
# ============================================================
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", linewidth=2)
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Frequency")
plt.title("Calibration Curve")
plt.tight_layout()
plt.savefig("figure_calibration.png", dpi=300)
plt.close()

# ============================================================
# 4. Confusion Matrix
# ============================================================
y_pred = (y_pred_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix (Threshold = 0.5)")
plt.tight_layout()
plt.savefig("figure_confusion_matrix.png", dpi=300)
plt.close()

# ============================================================
# 5. Permutation Feature Importance
# ============================================================
result = permutation_importance(
    model, X_test, y_test, n_repeats=20, random_state=42
)

importances = result.importances_mean
indices = np.argsort(importances)

plt.figure(figsize=(8, 5))
plt.barh(np.array(features)[indices], importances[indices])
plt.xlabel("Mean Importance")
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.savefig("figure_feature_importance.png", dpi=300)
plt.close()

# ============================================================
# 6. Predictor Distributions
# ============================================================
for col in features:
    plt.figure(figsize=(6, 4))
    sns.histplot(data=df, x=col, hue=target, kde=True, stat="density")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.savefig(f"figure_dist_{col}.png", dpi=300)
    plt.close()
